---

Image datasets and measurement

---

In [1]:
# autoload
%load_ext autoreload
%autoreload 2

# Load PGL libraries and start a PGL window
from pgl import pgl
from pgl.pglImage import pglImageDatabase, pglImageDatabaseWithManifest, pglImage
from pgl.pglMessages import pglMessages
from pgl.pglExperiment import pglTask, pglExperiment
from pgl.pglParameter import pglParameter, pglParameterBatch
import numpy as np

pgl = pgl()

# close any existing windows
pgl.cleanUp()

================================ pglBase: init =================================
(pgl) mglMetal error log can be viewed in MacOS Console app by searching for PROCESS mglMetal or in a terminal with:
      log stream --level info --process mglMetal
(pgl) To search for something specifc, e.g. messages from mglMovie:
      log stream --predicate 'eventMessage CONTAINS "mglMovie"' --style syslog --level info
(pgl:checkOS) Python version: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:35:20) [Clang 16.0.6 ]
(pgl:checkOS) Running on MacBook Pro (MacBookPro18,3) with macOS version: 26.6.1
(pgl:checkOS) Apple M1 Pro Cores: 8 (6 Performance and 2 Efficiency) Memory: 32 GB
(pgl:checkOS) GPU: Apple M1 Pro (Built-In) 14 cores, Metal 4 support
(pgl:checkOS)   Color LCD [Main Display]: 3024 x 1964 Retina (Built-in Liquid Retina XDR Display) GammaTable size: 1024
(pglBase) Main library instance created
(pglBase:shutdownAll) Shutting down mglMetal process: 14534
(pglBase:removeOrphanedSocket

---

Load the image database

---

In [ ]:
# load the database of images. Will check in directory for image formats that PIL
# knows about and make a list. This does not load the images, or check to see if they are valid
#imdb = pglImageDatabaseWithManifest(dataPath="ssh://justin@lagavulin/Users/justin/Desktop/NSD_shared1000")
imdb = pglImageDatabaseWithManifest(dataPath="ssh://justin@lagavulin/Users/justin/Desktop/things_200_img_12reps",filenameColumn="image_filename",indexColumn="test_image_nr",captionColumn="concept")

---

Print and display images

---

In [ ]:
# display a single image
#imdb.images[111].display()

# print image metadata one-by-one, this may take some time because it 
# has to open each file 
#imdb.print() 
imdb.print()

---

Display image dataset in a dialog

---

In [ ]:
pgl.traitsDialog(imdb)

In [ ]:
img=imdb.getImage(0)

---

Make a task to display images

---

In [ ]:
class pglImageTask(pglTask):
    
    ########################
    def __init__(self, pgl):
        super().__init__(pgl)
        
        # set task parameters, these will automatically be saved in the settings file
        self.settings.taskName = "Image Task"
        self.settings.nTrials = 2
        
        # fixed parameters, these will automatically be saved in the settings file
        self.settings.fixedParameters = {
            #'imagesDirectory': "ssh://justin@lagavulin/Users/justin/Desktop/NSD_shared1000",
            #'manifestColumnNames': {'filenameColumn':"filename",'indexColumn':"index",'captionColumn':"caption_1"},
            'imagesDirectory': "ssh://justin@lagavulin/Users/justin/Desktop/things_200_img_12reps",
            'manifestColumnNames': {'filenameColumn':"image_filename",'indexColumn':"test_image_nr",'captionColumn':"concept"},
            'imdb': None,
            'nImages': 10,
            'imdbCatch': None,
            'catchImagesDirectory': "ssh://justin@lagavulin/Users/justin/Desktop/things_200_catch_img_12reps",
            'catchManifestColumnNames': {'filenameColumn':"image_filename",'indexColumn':"catch_nr",'captionColumn':"original_filename"},
            'nCatchImages': 3,
            'imageSize': 18,
            'nImagesPerTrial': 10,
            'catchTrialEvery': None,
        }        
        p = self.settings.fixedParameters
        
        # set seglens, 
        # 1st segment is image display
        # 2nd segment is blank
        self.settings.seglen = [0.5, 0.5] * p['nImagesPerTrial']

        # initialize image database using parameters set from fixedParameters
        imdb = pglImageDatabaseWithManifest(
            p['imagesDirectory'],
            filenameColumn=p['manifestColumnNames']['filenameColumn'],
            indexColumn=p['manifestColumnNames']['indexColumn'],
            captionColumn=p['manifestColumnNames']['captionColumn'],
        )
        if imdb.nImages==0:
            pglMessages.warning(f"No images found in {p['imagesDirectory']}")
            return
        p['imdb'] = imdb
        
        # initialize catch image database using parameters set from fixedParameters
        imdbCatch = pglImageDatabaseWithManifest(
            p['catchImagesDirectory'],
            filenameColumn=p['catchManifestColumnNames']['filenameColumn'],
            indexColumn=p['catchManifestColumnNames']['indexColumn'],
            captionColumn=p['catchManifestColumnNames']['captionColumn'],
        )
        if imdbCatch.images==0:
            pglMessages.warning(f"No images found in {p['imagesDirectory']}")
            return
        p['imdbCatch'] = imdbCatch
        
        # preload images
        for iImage in range(p['nImages']):
            imdb.preloadImage(iImage)
        for iImage in range(p['nCatchImages']):
            imdbCatch.preloadImage(iImage)
            
        # add parameter for image number
        imageNum = pglParameterBatch('imageNum',np.arange(p['nImages']),batchSize=p['nImagesPerTrial'], catchTrialEvery=p['catchTrialEvery'])
        self.addParameter(imageNum)
        
        # set current image
        self.state.currentImage = None

    ########################
    def startSegment(self, startTime):
        '''
        Start a segment
        '''
        super().startSegment(startTime)
    
        # load the image
        if self.state.currentSegment % 2 == 0: 
            # get image database
            imdb = self.settings.fixedParameters['imdb']
            # get the current image number
            imageNums = self.currentParams['imageNum']
            if imageNums:
                # load an image
                imageNum = imageNums[int(self.state.currentSegment/2)]
                # get the image data
                img = imdb.getImage(imageNum)
                img.convert("RGB")
                print(f"img: {img}")
                # turn into a pglImage
                self.state.currentImage = self.pgl.imageCreate(np.array(img))
            else:
                self.state.currentImage = None
    ########################
    # updateScren
    ########################
    def updateScreen(self):
        '''
        update the screen
        '''
        if self.state.currentSegment % 2 == 0: 
            if self.state.currentImage:
                self.state.currentImage.display(height=self.settings.fixedParameters['imageSize'])
        
        # Draw ABC fixation cross from Thaler, Schütz, Goodale & Gegenfurtner (2013) Vision Research 76:31-42
        pgl.arc(0,0,0,0.3,stopAngle=2*np.pi,borderSize=0,color=0)
        pgl.rect(-0.3,0,width=0.6,height=0.15,color=1,hAlign='left',vAlign='center')
        pgl.rect(0,-0.3,width=0.15,height=0.6,color=1,hAlign='center',vAlign='top')
        pgl.arc(0,0,0,0.1,stopAngle=2*np.pi,borderSize=0,color=0)

        


---

Setup experiment

---

In [3]:
pgl.cleanUp()
#e = pglExperiment(pgl,settingsName='Cinema',experimentName='imageTask')
e = pglExperiment(pgl,experimentName='imageTask')

imageTask = pglImageTask(pgl)
e.addTask(imageTask)

(pglImageDatabaseWithManifest->pglImageDatabase:__init__) Found 200 image files in things_200_img_12reps
(pglImageDatabaseWithManifest:__init__) Sorted and set captions from manifest
(pglImageDatabaseWithManifest->pglImageDatabase:__init__) Found 200 image files in things_200_catch_img_12reps
(pglImageDatabaseWithManifest:__init__) Sorted and set captions from manifest
(pglImageFile:_loadImage) Loading image: ssh://justin@lagavulin//Users/justin/Desktop/things_200_img_12reps/alligator_14n.jpg
(pglImageFile:_loadImage) Loading image: ssh://justin@lagavulin//Users/justin/Desktop/things_200_img_12reps/altar_13s.jpg
(pglImageFile:_loadImage) Loading image: ssh://justin@lagavulin//Users/justin/Desktop/things_200_img_12reps/ashtray_14n.jpg
(pglImageFile:_loadImage) Loading image: ssh://justin@lagavulin//Users/justin/Desktop/things_200_img_12reps/axe_14n.jpg
(pglImageFile:_loadImage) Loading image: ssh://justin@lagavulin//Users/justin/Desktop/things_200_img_12reps/bamboo_13s.jpg
(pglImageFile

---

run experiment

---

In [4]:
e.initScreen()
e.run()

(pglBase:removeOrphanedSockets) No orphaned sockets found in /Users/justin/Library/Containers/gru.mglMetal/Data
================================= pglBase:open =================================
(pgl:_resolution:getResolution) Display 0/1: 1512x982 120Hz 32bits
(pglBase:getMetalAppName) Using latest build: /Users/justin/Library/Developer/Xcode/DerivedData/Build/Products/Release/mglMetal.app
(pgl->pglBase:open) Starting mglMetal application: /Users/justin/Library/Developer/Xcode/DerivedData/Build/Products/Release/mglMetal.app
(pgl->pglBase:open) Using socket with address: /Users/justin/Library/Containers/gru.mglMetal/Data/pglMetal.socket.20260815_225336.bN9CrVv0Fw
(pgl:_pglComm) .Connected to: /Users/justin/Library/Containers/gru.mglMetal/Data/pglMetal.socket.20260815_225336.bN9CrVv0Fw
(pgl:_resolution:getResolution) Display 0/1: 1512x982 120Hz 32bits
(pgl:pglTransform:pix2deg) xPix2Deg and yPix2Deg must be set before calling this function.
(pgl->pglMessages:oneTimeWarning) ⚠️ Need to fix

TypeError: '<' not supported between instances of 'NoneType' and 'int'

In [ ]:
e.tasks[0].settings.fixedParameters

In [ ]:
pgl.open(0)

In [ ]:
pgl.arc(0,0,0,0.3,stopAngle=2*np.pi,borderSize=0,color=0)
pgl.rect(-0.3,0,width=0.6,height=0.15,color=1,hAlign='left',vAlign='center')
pgl.rect(0,-0.3,width=0.15,height=0.6,color=1,hAlign='center',vAlign='top')
pgl.arc(0,0,0,0.1,stopAngle=2*np.pi,borderSize=0,color=0)

pgl.flush()